# 01 - Data Download and Benchmark Returns

This notebook downloads adjusted ETF prices, computes daily returns, and builds a simple benchmark return series used later in the project.


In [12]:
from pathlib import Path

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go

pd.options.display.float_format = "{:,.4f}".format

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BALANCED_BENCHMARK_WEIGHTS = {
    "SPY": 0.25,
    "EFA": 0.10,
    "EEM": 0.05,
    "IEF": 0.15,
    "TLT": 0.10,
    "LQD": 0.08,
    "HYG": 0.04,
    "GLD": 0.08,
    "DBC": 0.07,
    "VNQ": 0.05,
    "SHY": 0.03
}

default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


## 1. Download and clean adjusted prices

The universe is a small cross-asset ETF set covering equities, fixed income, credit, commodities, gold, real estate, and a short-duration Treasury cash proxy. Prices are downloaded on an adjusted basis so that dividends and splits are reflected in the return series.

The sample starts only once all ETFs have available prices. This avoids filling missing early histories, but it also means the ETF universe is chosen with hindsight and should be described as a research universe rather than a live investable universe selected in 2007.


In [13]:
tickers = [
    "SPY",  # US equities
    "EFA",  # developed ex-US equities
    "EEM",  # emerging markets
    "IEF",  # 7-10Y US Treasuries
    "TLT",  # long US Treasuries
    "LQD",  # investment-grade credit
    "HYG",  # high-yield credit
    "GLD",  # gold
    "DBC",  # commodities
    "VNQ",  # real estate
    "SHY"  # short-term Treasuries / cash proxy
]

start_date = "2007-01-01"
end_date = None  # Use None for latest available data; set a date to freeze the CV version.

prices_raw = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
)["Close"]

missing_tickers = [ticker for ticker in tickers if ticker not in prices_raw.columns]
if missing_tickers:
    raise ValueError(f"Missing tickers from download: {missing_tickers}")

prices = prices_raw.reindex(columns=tickers).dropna(how="any")
prices.index.name = "Date"

print("Price data shape:", prices.shape)
display(prices.head())


[*********************100%***********************]  10 of 11 completed

Price data shape: (4823, 11)


Ticker,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2007-04-11,101.3829,44.2195,26.9656,50.8629,48.2385,50.0664,31.6928,67.0800,20.6377,35.8069,57.7897
2007-04-12,101.8333,44.5147,27.4165,50.9122,48.2495,50.1417,31.7141,66.9900,20.7595,35.5632,57.7825
2007-04-13,102.2980,44.6624,27.5511,50.8258,48.1004,50.0570,31.6564,67.8400,20.8732,35.9648,57.7536
2007-04-16,103.2695,45.1336,27.8785,50.8813,48.3654,50.0994,31.6443,68.4000,20.7108,36.0100,57.7465
2007-04-17,103.5439,45.1393,27.7282,51.0974,48.6360,50.3725,31.6291,68.0000,20.4996,36.4793,57.8402


In [14]:
returns = prices.pct_change().dropna(how="any")
returns.index.name = "Date"

print("Daily returns shape:", returns.shape)
display(returns.head())

Daily returns shape: (4822, 11)


Ticker,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2007-04-12,0.0044,0.0067,0.0167,0.0010,0.0002,0.0015,0.0007,-0.0013,0.0059,-0.0068,-0.0001
2007-04-13,0.0046,0.0033,0.0049,-0.0017,-0.0031,-0.0017,-0.0018,0.0127,0.0055,0.0113,-0.0005
2007-04-16,0.0095,0.0106,0.0119,0.0011,0.0055,0.0008,-0.0004,0.0083,-0.0078,0.0013,-0.0001
2007-04-17,0.0027,0.0001,-0.0054,0.0042,0.0056,0.0055,-0.0005,-0.0058,-0.0102,0.0130,0.0016
2007-04-18,0.0012,-0.0004,-0.0074,0.0023,0.0050,0.0003,0.0003,0.0056,0.0004,-0.0062,0.0014


## 2. Benchmark portfolios

One transparent daily benchmark return series is created: a moderate risk cross-asset portfolio using all the ETFs of the universe. The benchmark is used as areference point, not as an optimized alternative.

In [15]:
benchmark_returns = returns @ pd.Series(BALANCED_BENCHMARK_WEIGHTS)
benchmark_returns.index.name = "Date"
benchmark_returns.name = "Balanced Benchmark Portfolio"

benchmark_returns.head()
benchmark_returns.shape

(4822,)

In [16]:
cumulative_benchmarks = (1.0 + benchmark_returns).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(x=cumulative_benchmarks.index, y=cumulative_benchmarks, mode="lines", name="Balanced Benchmark"))

fig.update_layout(
    **default_layout,
    title="Benchmark Portfolio Growth",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
)
fig.show()

## 3. Benchmark performance diagnostics

The metrics provide a baseline for later comparisons. Sharpe ratios are reported with a 0% risk-free rate for simplicity; this assumption should be stated explicitly in any interview discussion.


In [17]:
def performance_metrics(return_table, periods_per_year=252, risk_free_rate=0.0):
    r = return_table if isinstance(return_table, pd.DataFrame) else return_table.to_frame()
    r = r.dropna(how="all")

    ann_return = (1.0 + r).prod() ** (periods_per_year / len(r)) - 1.0
    ann_vol = r.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = (ann_return - risk_free_rate) / ann_vol

    cumulative = (1.0 + r).cumprod()
    drawdown = cumulative / cumulative.cummax() - 1.0

    return pd.DataFrame({
        "Annual Return": ann_return,
        "Annual Volatility": ann_vol,
        "Sharpe (rf=0)": sharpe,
        "Max Drawdown": drawdown.min(),
    })

In [18]:
benchmark_metrics = performance_metrics(benchmark_returns)

benchmark_metrics.style.format({
    "Annual Return": "{:.2%}",
    "Annual Volatility": "{:.2%}",
    "Sharpe (rf=0)": "{:.2f}",
    "Max Drawdown": "{:.2%}",
})

,Annual Return,Annual Volatility,Sharpe (rf=0),Max Drawdown
Balanced Benchmark Portfolio,7.00%,10.49%,0.67,-30.49%


In [19]:
drawdowns = cumulative_benchmarks / cumulative_benchmarks.cummax() - 1.0

y_bottom = drawdowns.min().min() * 1.10

fig = go.Figure()
fig.add_trace(go.Scatter(x=drawdowns.index, y=drawdowns, mode="lines"))

fig.update_layout(
    **default_layout,
    title="Benchmark Drawdowns",
    xaxis_title="Date",
    yaxis_title="Drawdown",
)
fig.update_yaxes(tickformat=".0%", range=[y_bottom, 0.02])
fig.show()

In [20]:
prices.to_csv(DATA_DIR / "prices.csv")
returns.to_csv(DATA_DIR / "returns.csv")
benchmark_returns.to_csv(DATA_DIR / "benchmark_returns.csv")

print("Saved price, return, and benchmark files to:", DATA_DIR.resolve())

Saved price, return, and benchmark files to: C:\Users\mmodr\Desktop\project-trend-invvol-strategy\data
